In [ ]:
import pandas as pd
import json

from curation_tools.curation_tools import (
    CuratedDataset,
    ObsSchema,
    VarSchema,
    Experiment,
    download_file,
    upload_parquet_to_bq
)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[
        logging.FileHandler("curation.log"),
        logging.StreamHandler(),  # keep console output too
    ],
    force=True,
)
# show all columns
pd.set_option('display.max_columns', None)

# Download data

In [ ]:
# data
download_file(
    url="https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE247599&format=file",
    dest_path='../non_curated/song_2025_jurkat_hiv.tar',
    unarchive=True
)
# guide-cell mapping
download_file(
    url="https://raw.githubusercontent.com/davidliwei/PS/refs/heads/main/datasets/HIV_Perturb-seq/BARCODE_H13Ld2EGFP.txt",
    dest_path='../supplementary/song_2025_jurkat_hiv/BARCODE_H13Ld2EGFP.txt',
    unarchive=False
)

Data is in three sets of files (mtx, barcodes, features) for each of the three conditions (NoDrug, LRA-Negative (no HIV expression)) and LRA-Positive (HIV expression). 

In [ ]:
import scanpy as sc
import scipy.io
import anndata as ad
import numpy as np

DATA_DIR = "../non_curated"

samples = {
    "LRA-Positive": "GSM7897841_JKLAT-LRA-Positive",
    "LRA-Negative": "GSM7897842_JKLAT-LRA-Negative",
    "NoDrug":       "GSM7897843_JKLAT-NoDrug",
}

def read_10x_mtx(data_dir, prefix):
    matrix   = scipy.io.mmread(f"{data_dir}/{prefix}_matrix.mtx.gz").T.tocsr()
    barcodes = pd.read_csv(f"{data_dir}/{prefix}_barcodes.tsv.gz", header=None)[0]
    features = pd.read_csv(
        f"{data_dir}/{prefix}_features.tsv.gz",
        sep="\t", header=None,
        names=["gene_id", "gene_name", "feature_type"],
    )
    obs = pd.DataFrame(index=barcodes)
    var = features.set_index("gene_id")
    return ad.AnnData(X=matrix, obs=obs, var=var)

adatas = {}
for condition, prefix in samples.items():
    adata = read_10x_mtx(DATA_DIR, prefix)
    adata.obs["condition"] = condition
    adatas[condition] = adata
    print(f"{condition}: {adata.shape}")

adata = ad.concat(adatas.values(), label="condition", keys=adatas.keys())
adata.var = adatas["LRA-Positive"].var  # restore var metadata lost during concat
print("\nConcatenated:", adata)


In [ ]:
# add cell barcode to obs
adata.obs['cell_barcode'] = adata.obs_names + '_' + adata.obs['condition'].astype(str)
adata.obs[['cell_barcode']]

In [ ]:
adata.obs

In [ ]:
adata.var

# Add available guide information to adata.obs

In [ ]:
guide_map_df = pd.read_csv("../supplementary/song_2025_jurkat_hiv/BARCODE_H13Ld2EGFP.txt", sep="\t")
guide_map_df['cell_barcode'] = guide_map_df['cell'].str.split('_').str[1]
guide_map_df['cell_barcode'] = guide_map_df['cell_barcode'] + '_' + guide_map_df['cell'].str.split('_').str[0].map({
    'pos': 'LRA-Positive',
    'nes': 'LRA-Negative',
    'nodrug': 'NoDrug'
})
# some cells have multiple guides - group by cell barcode and aggregate barcode, gene, sgrna together, separated by '|'
# then remove the cells with multiple guides
guide_map_df = guide_map_df.groupby('cell_barcode').agg({
    'barcode': lambda x: '|'.join(x),
    'gene': lambda x: '|'.join(x),
    'sgrna': lambda x: '|'.join(x)
}).reset_index()
# filter out cells with multiple guides
guide_map_df = guide_map_df[~guide_map_df['sgrna'].str.contains('\\|')]
# standardise controls
guide_map_df['gene'] = (
    guide_map_df['barcode']
    .replace(r'NegativeControl\d+', 'control_nontargeting', regex=True)
    .replace('HScontrol-AAVS1', 'control_gsh'))
# clean gene names
guide_map_df['gene'] = guide_map_df['gene'].str.split('-').str[1].fillna(guide_map_df['gene'])
# rename cols
guide_map_df = guide_map_df.rename(columns={'barcode': 'perturbation_name', 'sgrna': 'guide_sequence'})
guide_map_df

In [ ]:
# merge guide map with adata.obs; 
adata.obs = adata.obs.merge(guide_map_df, on='cell_barcode', how='left')
adata = adata[~adata.obs['guide_sequence'].isna()].copy()  # filter out cells without guide assignment

In [ ]:
adata.obs

# Save to a non-curated h5ad file

In [ ]:
adata.write_h5ad("../non_curated/h5ad/song_2025_jurkat_hiv.h5ad")
del(adata, guide_map_df, adatas)

# Initialise the dataset object

In [ ]:
noncurated_path = "../non_curated/h5ad/song_2025_jurkat_hiv.h5ad"
cur_data = CuratedDataset(
    obs_schema=ObsSchema,
    var_schema=VarSchema,
    exp_metadata_schema=Experiment,
    noncurated_path=noncurated_path
)

cur_data.load_data()

In [ ]:
cur_data.adata.obs

### Standardise perturbation targets

In [ ]:
cur_data.standardize_genes(
    slot='obs',
    input_column='gene',
    input_column_type='gene_symbol',
    multiple_entries=False
)

### Add `perturbed_target_number` column

In [ ]:
cur_data.count_entries(
    slot='obs',
    input_column='perturbed_target_symbol',
    count_column_name='perturbed_target_number',
    sep='|'
)

### Encode chromosomes as integers

In [ ]:
cur_data.chromosome_encoding()

In [ ]:
cur_data.adata.obs

### Add treatment information

In [ ]:
cur_data.adata.obs['treatment_label'] = cur_data.adata.obs['condition'].map({
    'LRA-Positive': 'HIV infection|phorbol 13-acetate 12-myristate|ionomycin',
    'LRA-Negative': 'HIV infection|phorbol 13-acetate 12-myristate|ionomycin',
    'NoDrug': 'HIV infection|dimethyl sulfoxide'
})

cur_data.adata.obs['treatment_id'] = cur_data.adata.obs['condition'].map({
    'LRA-Positive': 'EFO:0000764|CHEBI:37537|CHEBI:63954',
    'LRA-Negative': 'EFO:0000764|CHEBI:37537|CHEBI:63954',
    'NoDrug': 'EFO:0000764|CHEBI:28262'
})

### Add metadata

In [ ]:
cur_data.create_columns(
    overwrite=True,
    slot="obs",
    col_dict={
        "dataset_id": cur_data.dataset_id,
        "sample_id": range(1, cur_data.adata.obs.shape[0] + 1),
        # perturbation type
        "perturbation_type_label": "CRISPRn",
        "perturbation_type_id": None,
        "data_modality": "Perturb-seq",
        "significant": None,
        "significance_criteria": None,
        "score_interpretation": None,

        "technical_replicate": None,
        "biological_replicate": None,
        # treatment
        # "treatment_label": None,
        # "treatment_id": None,
        # model system
        "model_system_label": "cell_line",
        "model_system_id": None,
        "tissue": "blood",
        "cell_line_label": "JURKAT cell",
        "cell_type_label": "T cell",
        "disease_label": "T-cell childhood acute lymphocytic leukemia|HIV infectious disease",
        "disease_id": "MONDO:0000871|MONDO:0005109",

        "timepoint": "P7DT16H0M0S",
        "species": "Homo sapiens",
        "sex_label": "male",
        "sex_id": None,
        "developmental_stage_label": "adolescent",
        "developmental_stage_id": None,

        "study_title": "Decoding heterogeneous single-cell perturbation responses",
        "study_uri": "https://doi.org/10.1038/s41556-025-01626-9",
        "study_year": 2025,
        "first_author": "Bicna Song",
        "last_author": "Wei Li",

        "experiment_title": "Focused CRISPRn Perturb-seq of HIV latency regulators in Jurkat HIV model cell line under three conditions: DMSO-treated, PMA/I GFP⁺ (HIV reactivated), PMA/I GFP⁻ (HIV latent)",
        "experiment_summary": """
            A focused Perturb-seq screen targeting 10 known regulators of HIV transcription was performed in a latently infected Jurkat T-cell model.
            Cells were cultured for 7 days, after which they were subjected to either DMSO or phorbol 13-acetate 12-myristate/ionomycin (PMA/I) treatment.
            After 16 hours of treatment, PMA/I-treated cells were sorted into GFP-positive (HIV reactivated) and GFP-negative (HIV latent) populations, and all three conditions were profiled by single-cell RNA-seq.
            """,

        "number_of_perturbed_targets": len(set(cur_data.adata.obs['perturbed_target_coord'])),
        "number_of_perturbed_samples": cur_data.adata.obs.shape[0],

        "library_generation_type_id": "EFO:0022868",
        "library_generation_type_label": "endogenous",

        "library_generation_method_label": "SpCas9",
        "library_generation_method_id": "EFO:0022876",

        "enzyme_delivery_method_label": "lentivirus transduction",
        "enzyme_delivery_method_id": None,

        "library_delivery_method_label": "lentivirus transduction",
        "library_delivery_method_id": None,

        "enzyme_integration_state_label": "random locus integration",
        "enzyme_integration_state_id": None,

        "library_integration_state_label": "random locus integration",
        "library_integration_state_id": None,

        "enzyme_expression_control_label": "constitutive transgene expression",
        "enzyme_expression_control_id": None,

        "library_expression_control_label": "constitutive transgene expression",
        "library_expression_control_id": None,

        "library_name": "custom",
        "library_uri": None,

        "library_format_label": "pooled",
        "library_format_id": None,

        "library_scope_label": "focused",
        "library_scope_id": None,

        "library_perturbation_type_label": "knockout",
        "library_perturbation_type_id": None,

        "library_manufacturer": "Wei Li lab",
        "library_lentiviral_generation": "2",
        "library_grnas_per_target": "3",
        "library_total_grnas": str(cur_data.adata.obs['guide_sequence'].str.split('|').explode().nunique()),
        "library_total_variants": None,

        "readout_dimensionality_label": "high-dimensional assay",
        "readout_dimensionality_id": None,

        "readout_type_label": "transcriptomic",
        "readout_type_id": None,

        "readout_technology_label": "single-cell rna-seq",
        "readout_technology_id": None,

        "method_name_label": "Perturb-seq",
        "method_name_id": None,

        "method_uri": None,

        "sequencing_library_kit_label": "10x Genomics Single Cell 3-prime",
        "sequencing_library_kit_id": None,

        "sequencing_platform_label": "Illumina NovaSeq 6000",
        "sequencing_platform_id": None,

        "sequencing_strategy_label": "barcode sequencing",
        "sequencing_strategy_id": None,

        "software_counts_label": "CellRanger",
        "software_counts_id": None,

        "software_analysis_label": "Seurat",
        "software_analysis_id": None,

        "reference_genome_label": "GRCh38",
        "reference_genome_id": None,
        
        "license_label": "CC BY 4.0",
        "license_id": "SWO:1000065",

        "associated_datasets": json.dumps([
            {
                "dataset_accession": "GSE247599",
                "dataset_uri": "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE247599",
                "dataset_description": "Raw counts; matrix.mtx, features.tsv, barcodes.tsv",
                "dataset_file_name": "GSE247599_RAW.tar",
            }
        ])
    }
)

In [ ]:
cur_data.adata.obs

### Curate tissue information


In [ ]:
cur_data.standardize_ontology(
    input_column='tissue',
    column_type='term_name',
    ontology_type='tissue',
    overwrite=True
)

### Curate cell type information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_type_label',
    column_type='term_name',
    ontology_type='cell_type',
    overwrite=True
)

### Curate cell line information

In [ ]:
cur_data.standardize_ontology(
    input_column='cell_line_label',
    column_type='term_name',
    ontology_type='cell_line',
    overwrite=True
)

### Curate disease information

In [ ]:
# cur_data.standardize_ontology(
#     input_column='disease_label',
#     column_type='term_name',
#     ontology_type='disease',
#     overwrite=True
# )

### Match schema column order

In [ ]:
cur_data.match_schema_columns(slot='obs')

### Validate obs metadata

In [ ]:
cur_data.validate_data(slot='obs', verbose=True)

# VAR slot curation

### Standardise genes

In [ ]:
cur_data.show_var()

In [ ]:
# Keep only feature_type == Gene Expression
cur_data.adata = cur_data.adata[:, cur_data.adata.var['feature_type'] == 'Gene Expression'].copy()

In [ ]:
cur_data.create_columns(
    slot = 'var',
    col_dict={'gene_ensembl_id': cur_data.adata.var.index},
    overwrite=True
)

In [ ]:
cur_data.standardize_genes(
    slot="var",
    input_column="gene_ensembl_id",
    input_column_type="ensembl_gene_id",
    remove_version=False,
    multiple_entries=False
)

### Replace missing gene symbols with original gene names

In [ ]:
cur_data.adata.var['gene_symbol'] = cur_data.adata.var['gene_symbol'].fillna(cur_data.adata.var['gene_name'])

### Remove non ENSG entries from ensembl_gene_id column

In [ ]:
cur_data.adata.var.loc[~cur_data.adata.var['ensembl_gene_id'].str.startswith('ENSG'), 'ensembl_gene_id'] = np.nan

### Validate var metadata

In [ ]:
cur_data.validate_data(slot='var')

# Save the dataset

In [ ]:
cur_data.save_curated_data_h5ad()

In [ ]:
cur_data.save_curated_data_parquet(split_metadata=True, save_metadata_only=True)

# Upload to BigQuery

In [ ]:
upload_parquet_to_bq(
    parquet_path='../curated/parquet/song_2025_jurkat_hiv_curated_metadata.parquet',
    bq_dataset_id='prj-ext-dev-pertcat-437314.perturb_seq',
    bq_table_name='metadata',
    key_columns=['dataset_id', 'sample_id'],
    verbose=True
)

# Upload to GC Storage

In [ ]:
!gcloud storage cp ../curated/h5ad/song_2025_jurkat_hiv_curated.h5ad gs://perturbation-catalogue-lake/perturbseq/curated/